# Kaggriculture: submission-ready mixed-crop agent

This notebook builds and validates a Kaggriculture submission that follows the current competition contract:

- the archive contains **`main.py` at its root**;
- `main.py` exposes **`agent(obs)`**;
- every action uses the required `farmer` / `hands` / `market` schema;
- the submitted agent is self-contained and performs no network or filesystem access during an episode;
- the final notebook output is **`submission.tar.gz`**.

The supplied tutorial writes `submission.py` and exposes `melon_maxxer`, so its in-memory test does not validate the actual submission loader. This notebook fixes both issues and tests the file path plus the archive layout.

Official references: [competition overview](https://www.kaggle.com/competitions/kaggriculture/overview), [rules](https://www.kaggle.com/competitions/kaggriculture/rules).

## 1. Install the same environment version used for local validation

This installation is only for building/testing the notebook. The submitted `main.py` itself has no external dependencies.

In [1]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"

## 2. Write the required entrypoint

The policy uses the initially unlocked quadrant, hires three inexpensive daily hands, maintains a balanced 16-plot crop mix, sells in price-aware batches, and liquidates before the season ends. It is deterministic and stateless across episodes.

In [2]:
%%writefile main.py
"""Submission-ready Kaggriculture agent.

The agent is deliberately self-contained: it uses only the observation supplied by
the environment and performs no network, filesystem, or third-party-package work.
"""


PASS = ["PASS"]
DESIRED_HANDS = 3
MAX_MARKET_ORDERS = 10
LIQUIDATION_DAY = 27
FINAL_FARM_DAY = 28


# Current official environment constants used by the policy.  Keeping these local
# avoids relying on private implementation imports during Kaggle validation.
CROPS = {
    "WHEAT": {
        "seed_cost": 10,
        "first_yield_day": 2,
        "harvest_day": 4,
        "last_plant_day": 24,
        "ongoing": False,
    },
    "CARROT": {
        "seed_cost": 20,
        "first_yield_day": 2,
        "harvest_day": 3,
        "last_plant_day": 25,
        "ongoing": False,
    },
    "TOMATO": {
        "seed_cost": 50,
        "first_yield_day": 8,
        "harvest_day": 8,
        "last_plant_day": 17,
        "ongoing": True,
    },
    "MELON": {
        "seed_cost": 80,
        "first_yield_day": 10,
        # Daily watering reaches the six-unit cap at age 10.
        "harvest_day": 10,
        "last_plant_day": 18,
        "ongoing": False,
    },
}


# Sixteen plots in the initially unlocked NW quadrant.  A balanced crop mix
# spreads market risk while staying small enough for one farmer plus three hands.
PLOT_PLAN = (
    (1, 1, "WHEAT"),
    (2, 1, "CARROT"),
    (3, 1, "TOMATO"),
    (4, 1, "MELON"),
    (1, 2, "CARROT"),
    (2, 2, "TOMATO"),
    (3, 2, "MELON"),
    (4, 2, "WHEAT"),
    (1, 3, "TOMATO"),
    (2, 3, "MELON"),
    (3, 3, "WHEAT"),
    (4, 3, "CARROT"),
    (1, 4, "MELON"),
    (2, 4, "WHEAT"),
    (3, 4, "CARROT"),
    (4, 4, "TOMATO"),
)


# (normal batch, minimum acceptable price).  Forced liquidation ignores the
# threshold so that unsold shed inventory does not become worthless at game end.
SELL_RULES = {
    "WHEAT": (12, 17),
    "CARROT": (8, 24),
    "TOMATO": (5, 38),
    "MELON": (2, 125),
}


def _safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _step_toward(position, target):
    """Return one deterministic Manhattan step toward target."""
    x, y = position
    tx, ty = target
    dx = tx - x
    dy = ty - y
    if abs(dx) >= abs(dy) and dx:
        return ["EAST" if dx > 0 else "WEST"]
    if dy:
        return ["SOUTH" if dy > 0 else "NORTH"]
    return PASS


def _tile_task(tile, expected_crop, day, hour):
    """Return (priority, action) for one managed plot, or None."""
    crop_data = CROPS[expected_crop]

    if tile is None:
        if day <= crop_data["last_plant_day"] and hour <= 20:
            return 3, ["PLANT", expected_crop]
        return None

    if tile == "LOCKED" or not isinstance(tile, dict):
        return None

    kind = tile.get("kind")
    if kind == "WEED":
        return 2, ["DIG"]
    if kind != "PLANT":
        return None

    crop = tile.get("crop")
    actual = CROPS.get(crop)
    if actual is None:
        return 2, ["DIG"]

    age = day - _safe_int(tile.get("planted_day"), day)
    yield_units = _safe_int(tile.get("yield_units"), 0)
    watered = bool(tile.get("watered_today", False))

    # On the penultimate day, bank anything already harvestable.  Inventory is
    # dropped into the shed overnight and can then be sold throughout day 29.
    if day >= FINAL_FARM_DAY and yield_units > 0 and age >= actual["first_yield_day"]:
        return 0, ["HARVEST"]

    if actual["ongoing"]:
        if yield_units > 0 and age >= actual["first_yield_day"]:
            return 0, ["HARVEST"]
        if not watered:
            return 1, ["WATER"]
        return None

    # For one-time crops, water on the target harvest day before harvesting so
    # the final daily bonus is included.  If a harvest is already overdue, take
    # it immediately before per-turn decay can erode it.
    if yield_units > 0 and age >= actual["harvest_day"]:
        if age == actual["harvest_day"] and not watered:
            return 0, ["WATER"]
        return 0, ["HARVEST"]
    if not watered:
        return 1, ["WATER"]
    return None


def _build_tasks(farm, private, day, hour):
    """Build unique plot tasks without over-requesting any crop's seeds."""
    seeds_available = {
        crop: _safe_int((private.get("seeds", {}) or {}).get(crop), 0)
        for crop in CROPS
    }
    tasks = []

    for x, y, expected_crop in PLOT_PLAN:
        try:
            tile = farm["tiles"][y][x]
        except (IndexError, KeyError, TypeError):
            continue
        task = _tile_task(tile, expected_crop, day, hour)
        if task is None:
            continue
        priority, action = task
        if action[0] == "PLANT":
            if seeds_available[expected_crop] <= 0:
                continue
            seeds_available[expected_crop] -= 1
        tasks.append(
            {
                "priority": priority,
                "target": (x, y),
                "action": action,
            }
        )
    return tasks


def _assign_unit_actions(farm, private, day, hour):
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(pos) for pos in (farm.get("hands", []) or []))
    actions = [PASS for _ in positions]
    tasks = _build_tasks(farm, private, day, hour)
    remaining_units = set(range(len(positions)))

    # Global greedy matching: task urgency dominates, then travel distance.
    while tasks and remaining_units:
        choices = []
        for unit_index in remaining_units:
            ux, uy = positions[unit_index]
            for task_index, task in enumerate(tasks):
                tx, ty = task["target"]
                distance = abs(tx - ux) + abs(ty - uy)
                choices.append(
                    (
                        task["priority"],
                        distance,
                        task["target"][1],
                        task["target"][0],
                        unit_index,
                        task_index,
                    )
                )
        _, _, _, _, unit_index, task_index = min(choices)
        task = tasks.pop(task_index)
        remaining_units.remove(unit_index)
        if positions[unit_index] == task["target"]:
            actions[unit_index] = task["action"]
        else:
            actions[unit_index] = _step_toward(positions[unit_index], task["target"])

    return actions


def _sell_orders(private, market, day):
    shed = private.get("shed", {}) or {}
    prices = (market or {}).get("prices", {}) or {}
    shed_total = sum(max(0, _safe_int(v)) for v in shed.values())
    forced = day >= LIQUIDATION_DAY or shed_total >= 70
    orders = []

    for item in ("MELON", "TOMATO", "CARROT", "WHEAT"):
        held = _safe_int(shed.get(item), 0)
        if held <= 0:
            continue
        batch, threshold = SELL_RULES[item]
        price = _safe_int(prices.get(item), 0)
        if forced or price >= threshold:
            quantity = held if day >= LIQUIDATION_DAY else min(held, batch)
            orders.append(["SELL", item, quantity])
    return orders


def _seed_orders(farm, private, day, market_slots):
    if market_slots <= 0 or day >= FINAL_FARM_DAY:
        return []

    seeds = private.get("seeds", {}) or {}
    plants_by_crop = {crop: 0 for crop in CROPS}
    desired_by_crop = {crop: 0 for crop in CROPS}
    for x, y, crop in PLOT_PLAN:
        desired_by_crop[crop] += 1
        try:
            tile = farm["tiles"][y][x]
        except (IndexError, KeyError, TypeError):
            continue
        if isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile.get("crop") == crop:
            plants_by_crop[crop] += 1

    cash = float(farm.get("money", 0))
    reserve = 100.0
    orders = []
    # Cheap, fast crops first when cash is constrained; all four are stocked at
    # the start because the default $3,000 bank comfortably covers the plan.
    for crop in ("WHEAT", "CARROT", "TOMATO", "MELON"):
        data = CROPS[crop]
        if day > data["last_plant_day"] or len(orders) >= market_slots:
            continue
        missing = desired_by_crop[crop] - plants_by_crop[crop] - _safe_int(seeds.get(crop), 0)
        if missing <= 0:
            continue
        affordable = max(0, int((cash - reserve) // data["seed_cost"]))
        quantity = min(missing, affordable)
        if quantity <= 0:
            continue
        orders.append(["BUY_SEED", crop, quantity])
        cash -= quantity * data["seed_cost"]
    return orders


def _market_orders(farm, private, market, day, hour):
    orders = _sell_orders(private, market, day)

    # Hands hired at hour 0 appear after market processing and begin acting on
    # hour 1.  They disappear automatically at the end of each day.
    if hour == 0 and day <= FINAL_FARM_DAY:
        current_hands = len(farm.get("hands", []) or [])
        for _ in range(max(0, DESIRED_HANDS - current_hands)):
            if len(orders) >= MAX_MARKET_ORDERS:
                break
            orders.append(["HIRE"])

    free_slots = MAX_MARKET_ORDERS - len(orders)
    orders.extend(_seed_orders(farm, private, day, free_slots))
    return orders[:MAX_MARKET_ORDERS]


def agent(obs):
    """Kaggle entrypoint: return one action for every active unit and the market."""
    try:
        farms = obs.get("farms", []) or []
        player = _safe_int(obs.get("player"), 0)
        private = obs.get("private", {}) or {}
        if player < 0 or player >= len(farms):
            return {"farmer": PASS, "hands": [], "market": []}

        farm = farms[player]
        day = _safe_int(obs.get("day"), 0)
        hour = _safe_int(obs.get("hour"), 0)
        market = obs.get("market", {}) or {}

        market_orders = _market_orders(farm, private, market, day, hour)

        # Day 29 is reserved for liquidation.  Harvesting then would leave goods
        # in unit inventories with no later day on which to sell them.
        if day > FINAL_FARM_DAY:
            farmer_action = PASS
            hand_actions = [PASS for _ in (farm.get("hands", []) or [])]
        else:
            unit_actions = _assign_unit_actions(farm, private, day, hour)
            farmer_action = unit_actions[0] if unit_actions else PASS
            hand_actions = unit_actions[1:]

        return {
            "farmer": farmer_action,
            "hands": hand_actions,
            "market": market_orders,
        }
    except Exception:
        # A conservative fallback keeps a malformed/unexpected observation from
        # failing the validation episode.  Normal observations never use it.
        hands = []
        try:
            farms = obs.get("farms", []) or []
            player = _safe_int(obs.get("player"), 0)
            if 0 <= player < len(farms):
                hands = [PASS for _ in (farms[player].get("hands", []) or [])]
        except Exception:
            hands = []
        return {"farmer": PASS, "hands": hands, "market": []}

Writing main.py


## 3. Check the entrypoint and action contract

This catches the filename/function mismatch that the tutorial's in-memory callable test misses.

In [3]:
import ast
import importlib.util
import json
from pathlib import Path

main_path = Path("main.py")
assert main_path.is_file(), "main.py was not created"

tree = ast.parse(main_path.read_text(encoding="utf-8"), filename="main.py")
function_names = {node.name for node in tree.body if isinstance(node, ast.FunctionDef)}
assert "agent" in function_names, "main.py must expose def agent(obs)"

spec = importlib.util.spec_from_file_location("submission_agent", main_path)
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)
assert callable(submission_agent.agent)

tiles = [
    [None if x < 5 and y < 5 else "LOCKED" for x in range(10)]
    for y in range(10)
]
dummy_obs = {
    "player": 0,
    "day": 0,
    "hour": 0,
    "farms": [{
        "money": 3000,
        "tiles": tiles,
        "farmer": [4, 4],
        "hands": [],
        "unlocked_quadrants": ["NW"],
        "hires_today": 0,
    }],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"inventory": {}, "prices": {}},
    "town": {"unlocked_shops": []},
}

action = submission_agent.agent(dummy_obs)
assert set(action) == {"farmer", "hands", "market"}
assert isinstance(action["farmer"], list) and action["farmer"]
assert isinstance(action["hands"], list)
assert isinstance(action["market"], list) and len(action["market"]) <= 10
json.dumps(action)
print("Entrypoint and JSON action contract: OK")
print(action)

Entrypoint and JSON action contract: OK
{'farmer': ['PASS'], 'hands': [], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'WHEAT', 4], ['BUY_SEED', 'CARROT', 4], ['BUY_SEED', 'TOMATO', 4], ['BUY_SEED', 'MELON', 4]]}


## 4. Run full file-loader validation games

The first match mirrors Kaggle's self-play Validation Episode. The other matches check two built-in opponents. All games run the **`main.py` path**, not an in-memory function.

In [4]:
from kaggle_environments import make

opponents = ["main.py", "starter", "random"]
for index, opponent in enumerate(opponents):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 20260821 + index},
        debug=True,
    )
    env.run(["main.py", opponent])
    final = env.steps[-1]
    statuses = [state.status for state in final]
    rewards = [state.reward for state in final]
    assert statuses == ["DONE", "DONE"], (opponent, statuses)
    print(f"vs {opponent:8s}: statuses={statuses}, rewards={rewards}")

print("Full 720-turn file-loader validation: OK")

vs main.py : statuses=['DONE', 'DONE'], rewards=[20581.0, 20581.0]
vs starter : statuses=['DONE', 'DONE'], rewards=[21396.0, 3297.0]
vs random  : statuses=['DONE', 'DONE'], rewards=[24070.0, 0.0]
Full 720-turn file-loader validation: OK


## 5. Build and verify the submission archive

The member name must be exactly `main.py`, with no enclosing directory.

In [5]:
import hashlib
import tarfile
from pathlib import Path

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add("main.py", arcname="main.py", recursive=False)

with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getnames()
    assert members == ["main.py"], members
    archived_source = archive.extractfile("main.py").read()

assert archived_source == Path("main.py").read_bytes()
size_mib = archive_path.stat().st_size / (1024 * 1024)
assert size_mib < 100, f"Archive is too large: {size_mib:.2f} MiB"
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()

print(f"Created: {archive_path.resolve()}")
print(f"Members: {members}")
print(f"Size: {size_mib:.4f} MiB")
print(f"SHA-256: {sha256}")

Created: /kaggle/working/submission.tar.gz
Members: ['main.py']
Size: 0.0036 MiB
SHA-256: 19779a3b38ea63e0b6cdcfb1fcf797599050146076741396784afe82b785496b


## 6. Submit

1. Attach the **Kaggriculture** competition to this notebook and accept the competition rules.
2. Run all cells and save a successful notebook version.
3. Click **Submit to competition** and select the notebook output `submission.tar.gz`.

CLI equivalent:

```bash
kaggle competitions submit kaggriculture \
  -k YOUR_USERNAME/YOUR_NOTEBOOK_SLUG \
  -f submission.tar.gz \
  -v NOTEBOOK_VERSION \
  -m "mixed crop submission-ready agent"
```

After submission, confirm that the self-play Validation Episode finishes without an `Error` status.